## SFT / PEFT training (Colab)

Trains a model on the synthetic NAV4RAIL dataset.

Assumes you executed `00_colab_setup.ipynb` first (deps + `sys.path`).

Covers:
- Baseline SFT
- LoRA/QLoRA (PEFT)
- Loss masking (loss only on XML completion tokens)


In [ ]:
from pathlib import Path
import sys

BENCHMARK_ROOT = Path('/content/nav4rails/repositories/FineTuningOnTelecomCluster/benchmark').resolve()
if str(BENCHMARK_ROOT) not in sys.path:
    sys.path.insert(0, str(BENCHMARK_ROOT))

SCRIPTS_DIR = BENCHMARK_ROOT / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print('Benchmark root:', BENCHMARK_ROOT)


In [ ]:
# 1) Generate a synthetic dataset JSONL

import json
from src.data.synthetic_generator import iter_dataset

dataset_path = BENCHMARK_ROOT / 'data' / 'dataset_synthetic.jsonl'
dataset_path.parent.mkdir(parents=True, exist_ok=True)
rows = list(iter_dataset(200))
dataset_path.write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in rows) + '\n', encoding='utf-8')
print('wrote:', dataset_path, 'rows:', len(rows))


In [ ]:
# 2) Run SFT (LoRA) using the unified runner

from src.config_loader import load_experiment_config
from src.evaluation.runner import ExperimentRunner

cfg = load_experiment_config(BENCHMARK_ROOT / 'configs' / 'sft_lora.yaml')
# Put colab artifacts under runs/colab_* to avoid clobbering slurm outputs
cfg.output_root = str(BENCHMARK_ROOT / 'runs' / 'colab_sft_lora')

runner = ExperimentRunner(cfg)

# Load dataset rows
from _jsonl import read_jsonl
train_rows = list(read_jsonl(dataset_path))

res = runner.run_sft_experiment(train_rows=train_rows)
res